# 12장. 나만의 이야기 챗봇

요청을 쓰면 이야기를 들려주는 챗봇을 완성해요. **GPU가 없어도 됩니다.** T4 GPU를 연결하면 글을 조금 더 빨리 만들어요. 이야기는 실행할 때마다, 그리고 쓰는 `sft.pt`에 따라 달라지므로 책과 같은 글이 나오지 않아도 괜찮습니다.

드라이브에 11장의 `sft.pt`가 있으면 그것을, 없으면 저자의 체크포인트를 씁니다.


### 준비


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from modu_llm import Vocab, load_checkpoint, get_file, mount_drive
from modu_llm import generate, format_pair, get_device

device = get_device()
vocab = Vocab.build()
folder = mount_drive()

model, _, step, _ = load_checkpoint(get_file("sft.pt"), device)
model.eval()
print(f"파인튜닝 {step}단계 모델")


## 12.1 요청을 형식에 맞게 감싸기

모델은 11장에서 배운 형식으로만 요청을 알아들어요. 사용자가 쓴 요청을 그 형식으로 감싸 줍니다.

**핵심 코드**: `format_pair(request)`


In [ ]:
question = format_pair("숲속에서 토끼 보리가 나오는 이야기를 들려줘")
print(repr(question))


## 12.2 chat 함수 만들기

요청을 감싸고, 이야기를 만들고, `§`에서 멈추는 일을 함수 하나로 묶어요. 사전에 없는 글자가 요청에 있으면 알려 줍니다.

**핵심 코드**: `generate(model, vocab, format_pair(request), ...)`


In [ ]:
def chat(request, temperature=0.8):
    unknown = vocab.unknown_chars(request)
    if unknown:
        missing = " ".join(unknown)
        note = f"(사전에 없어 건너뛴 글자: {missing})\n\n"
    else:
        note = ""
    question = format_pair(request)
    story = generate(model, vocab, question, max_new_tokens=400,
                     temperature=temperature, top_k=20)
    return note + story.strip()

print(chat("강아지 뭉치가 부엌에서 겪은 이야기를 들려줘"))


사전에 없는 글자가 들어간 요청도 넣어 봅니다.


In [ ]:
print(chat("강아지 똠이가 나오는 이야기를 들려줘")[:60])


## 12.3 대화 화면 만들기

Gradio로 입력창이 있는 화면을 만들어요. 셀을 실행하면 노트북 안에 화면이 나타나고, 다른 사람에게 보낼 수 있는 임시 주소도 생깁니다.

**핵심 코드**: `gr.Interface(fn=chat, ...).launch()`


In [ ]:
%pip install -q gradio
import gradio as gr

request_box = gr.Textbox(label="요청", value="여름날 바닷가에서 토끼가 나오는 이야기를 들려줘")
temp_slider = gr.Slider(0.3, 1.5, value=0.8, label="온도")
story_box = gr.Textbox(label="이야기", lines=12)

demo = gr.Interface(
    fn=chat,
    inputs=[request_box, temp_slider],
    outputs=story_box,
    title="나만의 이야기 챗봇",
    flagging_mode="never",      # 결과를 파일로 모으는 Flag 버튼은 쓰지 않음
)
demo.launch()


Gradio가 잘 되지 않으면 아래 셀로도 대화할 수 있어요. 빈칸으로 엔터를 치면 끝납니다.


In [ ]:
while True:
    request = input("요청: ")
    if request == "":
        break
    print(chat(request))
    print()


## 12.4 작은 모델의 한계 알기

이 챗봇은 어린이 이야기만 배웠어요. 배운 범위를 벗어난 요청을 넣어 보세요. 어떤 요청에 무너지나요?


In [ ]:
hard_requests = ["우주선을 타고 화성에 가는 이야기를 들려줘",
                 "대한민국의 수도는 어디야?",
                 "1 더하기 1은?",
                 "Tell me a story about a cat."]
for r in hard_requests:
    story = chat(r)
    print("요청:", r)
    print(story[:200])
    print("-" * 40)


## 12.5 ChatGPT는 여기서 무엇을 더 할까

| | 이 책의 챗봇 | 실제 ChatGPT 같은 서비스 |
|---|---|---|
| 토큰 | 글자 하나 = 토큰 하나, 2,453자 | 하위 단어 토큰, 수만~수십만 개 |
| 크기 | 약 600만 개 파라미터 | 수십억~수천억 개 이상 |
| 사전학습 데이터 | 어린이 이야기 20만 편 | 웹·책·코드 등 수조 개 토큰 |
| 파인튜닝 | 요청-이야기 쌍 | 다양한 대화 데이터 + 사람의 선호를 반영한 학습 |
| 안전장치 | 없음 | 해로운 요청 거절, 검색이나 계산기 같은 도구 연결 등 |

그래도 **다음 토큰을 맞히도록 사전학습하고, 요청-응답 형식으로 파인튜닝한다**는 뼈대는 같아요. 여러분은 그 뼈대를 처음부터 직접 만들었습니다.


표의 첫 줄, 토큰을 직접 비교해 봐요. 실제 LLM의 토크나이저를 허깅페이스에서 내려받아 같은 요청을 나눠 봅니다. transformers 라이브러리는 Colab에 깔려 있어요.


In [ ]:
from transformers import AutoTokenizer

qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")    # 실제 LLM의 토크나이저

text = "숲속에서 토끼 보리가 나오는 이야기를 들려줘"
ids = qwen.encode(text)
pieces = []
for i in ids:
    pieces.append(qwen.decode([i]))
print(f"사전 크기: {len(qwen):,}개")
print(f"토큰 {len(ids)}개:", pieces)


영어 중심으로 만든 GPT-2의 토크나이저와도 비교해 봐요. 우리 글자 사전도 encode가 있으니 같은 반복문에 넣을 수 있어요.


In [ ]:
gpt2 = AutoTokenizer.from_pretrained("openai-community/gpt2")
english = "Tell me a story about a rabbit in the forest."

for name, tok in [("우리 글자 사전", vocab), ("Qwen2.5", qwen), ("GPT-2", gpt2)]:
    ko = len(tok.encode(text))
    en = len(tok.encode(english))
    print(f"{name}: 한국어 {ko}개, 영어 {en}개")


## 12.6 공개 모델 안에서 우리 부품 찾기

12-8의 토크나이저와 짝인 공개 모델 **Qwen2.5-0.5B**를 내려받아 안을 열어 봐요. 파라미터가 약 5억 개인 작은 LLM(SLM)입니다.
이름과 크기는 달라도 3~7장에서 직접 만든 부품이 그대로 들어 있어요. 처음 내려받을 때 1GB쯤 받느라 1~2분 걸립니다.

**핵심 코드**: `print(slm)`


In [ ]:
from transformers import AutoModelForCausalLM

slm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")   # 12-8 토크나이저와 짝인 모델
print(slm)


7.4에서 쓴 파라미터 세기로 우리 GPT와 크기를 비교해요.


In [ ]:
from modu_llm import count_params

block = slm.model.layers[0]
print(f"우리 GPT: {count_params(model):,}")
print(f"Qwen2.5: {count_params(slm):,}")
print(f"  글자 표: {count_params(slm.model.embed_tokens):,}")
print(f"  블록 하나: {count_params(block):,} (어텐션 {count_params(block.self_attn):,}, MLP {count_params(block.mlp):,})")
print(f"  블록 수: {len(slm.model.layers)}")


다음 토큰의 확률도 10장과 같은 계산으로 볼 수 있어요. 10.6에서 우리 모델이 답하지 못한 질문을 넣어 봅니다.


In [ ]:
from torch.nn import functional as F

for prompt in ["대한민국의 수도는", "토끼는 기분이 좋아서 깡충깡충 뛰었어"]:
    ids = torch.tensor([qwen.encode(prompt)])
    with torch.no_grad():
        logits = slm(ids).logits
    probs = F.softmax(logits[0, -1], dim=-1)          # 10장과 같은 계산
    line = f"[{prompt}]"
    for i in torch.topk(probs, 3).indices.tolist():
        line = line + f"  [{qwen.decode([i])}] {probs[i].item() * 100:.0f}%"
    print(line)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

요청 표현을 바꿔 보세요("들려줘" → "써 줘", 이름만 쓰기 등). `chat(r, temperature=0.3)`처럼 온도도 바꿔 보세요. 모델은 어디까지 알아듣나요?


In [ ]:
play_requests = ["보리 이야기", "보리가 나오는 이야기", "토끼 보리가 숲속에서 겪은 이야기를 들려줘"]
for r in play_requests:
    story = chat(r)
    print(r, "→", story[:120])


## 실제 LLM에서는

- 실제 챗봇은 이전 대화도 함께 입력해 맥락을 이어 가요. 이 챗봇은 요청 하나만 보고 답합니다.
- 응답을 만드는 대로 화면에 흘려 보여 주는 스트리밍, 앞 토큰의 키와 값을 저장해 두는 KV 캐시, 여러 사용자의 요청을 한꺼번에 처리하는 기술도 더해집니다.
- 여러분의 글로 모델을 더 학습시키고 싶다면 11장의 방법을 그대로 쓰면 돼요. 다만 새 글만 학습시키면 모델이 원래 하던 일을 잊어버립니다(**파국적 망각**). 그래서 실제로는 원래 학습 데이터를 섞어서 함께 학습시켜요.
